# 09 - Interpolação

Atividades da Semana 9 resolvidas seguindo os enunciados e resultados de referência do professor.

## Preparação — imports e caminho do projeto

A busca pela pasta `utils` funciona tanto quando o kernel inicia na raiz do repositório quanto dentro de `notebooks`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "utils" / "algoritmos.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Não encontrei utils/algoritmos.py. Abra o notebook dentro do repositório cnum.")
sys.path.insert(0, str(ROOT))

from utils.algoritmos import newton, lagrange, polinomial

print("Raiz do projeto:", ROOT)


## Atividade 1 — Newton, Lagrange, polinomial e spline cúbica

Pontos: `(-2, -47)`, `(0, -3)`, `(1, 4)` e `(2, 41)`. A spline natural é calculada com `bc_type="natural"`, conforme a referência do professor.

In [ ]:
x = np.array([-2.0, 0.0, 1.0, 2.0])
y = np.array([-47.0, -3.0, 4.0, 41.0])
xr = np.array([-1.0, 0.5, 1.5])

coef_newton = newton(x, y)
coef_lagrange = lagrange(x, y)
coef_polinomial = polinomial(x, y)
spline = CubicSpline(x, y, bc_type="natural")

print("Coeficientes Newton:     ", coef_newton)
print("Coeficientes Lagrange:   ", coef_lagrange)
print("Coeficientes polinomial: ", coef_polinomial)
print("\nValores interpolados:")
for xi, yn, yl, yp, ys in zip(
    xr,
    np.polyval(coef_newton, xr),
    np.polyval(coef_lagrange, xr),
    np.polyval(coef_polinomial, xr),
    spline(xr),
):
    print(f"x={xi:>4.1f} | Newton={yn: .6f} | Lagrange={yl: .6f} | Polinomial={yp: .6f} | Spline natural={ys: .6f}")

xx = np.linspace(x.min() - 0.25, x.max() + 0.25, 400)
plt.figure(figsize=(9, 5))
plt.plot(xx, np.polyval(coef_newton, xx), label="Newton")
plt.plot(xx, np.polyval(coef_lagrange, xx), "--", label="Lagrange")
plt.plot(xx, np.polyval(coef_polinomial, xx), ":", label="Polinomial")
plt.plot(xx, spline(xx), linewidth=2, label="Spline cúbica natural")
plt.scatter(x, y, color="black", zorder=5, label="Pontos")
plt.xlabel("x"); plt.ylabel("f(x)")
plt.title("Atividade 1 — Interpolação")
plt.grid(True); plt.legend(); plt.show()


## Atividade 2 — Interpolação e spline cúbica

Pontos: `(-1, 1.25)`, `(0.5, 0.5)`, `(1, 1.25)` e `(1.25, 1.8125)`. Usa-se a spline cúbica padrão (*not-a-knot*), que reproduz o polinômio quadrático dos dados.

In [ ]:
x2 = np.array([-1.0, 0.5, 1.0, 1.25])
y2 = np.array([1.25, 0.5, 1.25, 1.8125])
xr2 = np.array([-0.5, 0.0, 0.25])

c2n = newton(x2, y2)
c2l = lagrange(x2, y2)
c2p = polinomial(x2, y2)
s2 = CubicSpline(x2, y2)  # condição padrão not-a-knot

print("Polinômio esperado: f(x) = x² + 0.25")
for xi, yn, yl, yp, ys in zip(
    xr2, np.polyval(c2n, xr2), np.polyval(c2l, xr2),
    np.polyval(c2p, xr2), s2(xr2)
):
    print(f"x={xi:>4.2f} | Newton={yn:.6f} | Lagrange={yl:.6f} | Polinomial={yp:.6f} | Spline={ys:.6f}")

xx2 = np.linspace(x2.min(), x2.max(), 400)
plt.figure(figsize=(9, 5))
plt.plot(xx2, np.polyval(c2n, xx2), label="Newton")
plt.plot(xx2, np.polyval(c2l, xx2), "--", label="Lagrange")
plt.plot(xx2, np.polyval(c2p, xx2), ":", label="Polinomial")
plt.plot(xx2, s2(xx2), linewidth=2, label="Spline cúbica")
plt.scatter(x2, y2, color="black", zorder=5, label="Pontos")
plt.xlabel("x"); plt.ylabel("f(x)")
plt.title("Atividade 2 — Interpolação")
plt.grid(True); plt.legend(); plt.show()


## Atividade 3 — Curva de magnetização

(a) Estimar a indução magnética em `H = 0 A/m`. (b) Estimar o campo magnético em `T = 8.5 s`. A spline cúbica natural é usada para comparação, como indicado na resolução do professor.

In [ ]:
def comparar_interpoladores(x, y, ponto, titulo):
    cn = newton(x, y)
    cl = lagrange(x, y)
    cp = polinomial(x, y)
    cs = CubicSpline(x, y, bc_type="natural")
    valores = {
        "Newton": float(np.polyval(cn, ponto)),
        "Lagrange": float(np.polyval(cl, ponto)),
        "Polinomial": float(np.polyval(cp, ponto)),
        "Spline natural": float(cs(ponto)),
    }
    print(titulo)
    for nome, valor in valores.items():
        print(f"{nome}: {valor:.6f}")
    xx = np.linspace(np.min(x), np.max(x), 500)
    plt.figure(figsize=(9, 5))
    plt.plot(xx, np.polyval(cn, xx), label="Newton")
    plt.plot(xx, np.polyval(cl, xx), "--", label="Lagrange")
    plt.plot(xx, np.polyval(cp, xx), ":", label="Polinomial")
    plt.plot(xx, cs(xx), linewidth=2, label="Spline natural")
    plt.scatter(x, y, color="black", zorder=5, label="Medições")
    plt.scatter([ponto], [valores["Spline natural"]], marker="x", s=80, label="Ponto estimado")
    plt.title(titulo); plt.xlabel("Variável independente"); plt.ylabel("Variável dependente")
    plt.grid(True); plt.legend(); plt.show()
    return valores

# (a) Curva de desmagnetização: B em função de H
H = np.array([-50.0, -5.0, 5.0, 75.0])
B = np.array([-300.0, -50.0, 180.0, 350.0])
resultado_a = comparar_interpoladores(H, B, 0.0, "Atividade 3(a) — B para H=0 A/m")

# (b) Campo magnético em função do tempo
T = np.arange(11, dtype=float)
campo = np.array([80.0, -60.0, 40.0, -30.0, 20.0, -10.0, 5.0, -2.5, 1.25, -0.625, 0.3125])
resultado_b = comparar_interpoladores(T, campo, 8.5, "Atividade 3(b) — H para T=8.5 s")


## Observação sobre os resultados

Na atividade 3(b), os polinômios globais de grau 10 podem oscilar bastante (fenômeno de Runge). A spline natural é uma interpolação local e pode produzir uma estimativa muito diferente.

In [ ]:
# Verificação final: os três métodos polinomiais devem coincidir nos nós.
for nome, xxv, yyv in [
    ("Atividade 1", x, y),
    ("Atividade 2", x2, y2),
    ("Atividade 3(a)", H, B),
    ("Atividade 3(b)", T, campo),
]:
    for metodo in (newton, lagrange, polinomial):
        assert np.allclose(np.polyval(metodo(xxv, yyv), xxv), yyv, atol=1e-7)
    print(nome, "— interpolação nos pontos originais: OK")
